# 27 - Multi-Algorithm Comparison Analysis

Triggered by external review feedback (8+ years HVAC/FDD field experience): compare Random Forest against Logistic Regression, XGBoost, and a shallow neural net (MLP) per fault, using the same 5-fold `TimeSeriesSplit` standard as every other classifier metric in this project.

This notebook only visualizes results already computed by `ml/src/models/compare_algorithms.py` and saved to `MODEL_COMPARISON_RESULTS.json` - it does not re-run any training. Full methodology and per-fault decisions are documented in `ml/MODEL_COMPARISON_LOG.md`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_PATH = Path("../MODEL_COMPARISON_RESULTS.json")
with open(RESULTS_PATH) as f:
    results = json.load(f)

print(f"Loaded comparison results for {len(results)} faults")

In [ ]:
rows = []
for fault, algos in results.items():
    for algo, res in algos.items():
        rows.append({
            "fault": fault,
            "algorithm": algo,
            "recall_lo": res["recall_range"][0],
            "recall_hi": res["recall_range"][1],
            "precision_lo": res["precision_range"][0],
            "precision_hi": res["precision_range"][1],
            "tier": res["acceptance_tier"],
        })

df = pd.DataFrame(rows)
df["recall_mid"] = (df["recall_lo"] + df["recall_hi"]) / 2
df["precision_mid"] = (df["precision_lo"] + df["precision_hi"]) / 2
df

In [ ]:
ALGORITHMS = ["logistic_regression", "random_forest", "mlp", "xgboost"]
TIER_COLORS = {
    "Usable": "#2ca02c",
    "Usable with caveat": "#ff9f1c",
    "Not production-usable": "#d62728",
}

def plot_metric(metric_prefix, title):
    faults = list(results.keys())
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True)
    axes = axes.flatten()

    for ax, fault in zip(axes, faults, strict=False):
        fault_df = df[df["fault"] == fault].set_index("algorithm").reindex(ALGORITHMS)
        x = np.arange(len(ALGORITHMS))
        mid = fault_df[f"{metric_prefix}_mid"]
        lo = fault_df[f"{metric_prefix}_lo"]
        hi = fault_df[f"{metric_prefix}_hi"]
        err = np.abs(np.vstack([mid - lo, hi - mid]))
        colors = [TIER_COLORS.get(t, "#888888") for t in fault_df["tier"]]

        ax.bar(x, mid, yerr=err, capsize=4, color=colors, alpha=0.85)
        ax.axhline(0.90, color="green", linestyle="--", linewidth=0.8)
        ax.axhline(0.65, color="orange", linestyle="--", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(ALGORITHMS, rotation=30, ha="right")
        ax.set_title(fault)
        ax.set_ylim(0, 1.05)

    axes[0].set_ylabel(f"{title} (range across TimeSeriesSplit folds)")
    fig.suptitle(f"{title} by algorithm per fault - bar=midpoint, whiskers=fold min/max, dashed lines=acceptance-criteria floors")
    plt.tight_layout()
    return fig

fig_recall = plot_metric("recall", "Recall")
fig_recall.savefig("../MODEL_COMPARISON_RECALL_CHART.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig_precision = plot_metric("precision", "Precision")
fig_precision.savefig("../MODEL_COMPARISON_PRECISION_CHART.png", dpi=150, bbox_inches="tight")
plt.show()

## Conclusion

Of six faults compared across four algorithms each, exactly **one** (`evaporator_fouling`) has real, evidence-based justification to change algorithms:

- Currently shipped Random Forest has a documented, known weak recall floor here (0.76 -> 0.41 degradation in earlier evaluation).
- XGBoost resolves this directly: stable ~0.95 recall in every fold, crossing fully into the "Usable" tier.
- MLP scored marginally higher (0.997-0.999 recall) but was **not** chosen - it would be the only model in the fleet requiring a persisted `StandardScaler` at inference time, and tree models get fast, exact SHAP `TreeExplainer` support for the already-planned feature-importance work, while MLP needs slower/approximate explainers.

**Every other fault**: all four candidates land in the same acceptance tier as the currently shipped Random Forest, with overlapping, noise-level differences. No evidence supports switching algorithms for condenser fouling, liquid-line restriction, suction-line restriction, or overcharge - and for overcharge specifically, none of today's alternate configs even beat the currently deployed model's real 0.97-1.00 recall.

Full methodology, deliberate scope boundaries, and fold-level verification notes are in `ml/MODEL_COMPARISON_LOG.md`.